# COMP47590 Advanced Machine Learning

## Assignment: Learning to Walk
Uses the PPO actor-critic method to train a neural network to allow a simple robot to learn to walk using the **Bipedal Walker** environment from the `gymnasium` package (https://gymnasium.farama.org/environments/box2d/bipedal_walker/). This assignment uses a custom version of the environment that includes some small modifications, `BipedalWalkerUCD`.

![Bipedal Walker](bipedal_walker.gif)

The **action** space is continuous where the actions are motor speed values in the [-1, 1] range for each of the 4 joints at both hips and knees.

The **observation** is a state vector consisting of hull angle speed, angular velocity, horizontal speed, vertical speed, position of joints and joints angular speed, legs contact with ground, and 10 lidar rangefinder measurements. There are no coordinates in the state vector.

The basic **reward** provides a small positive rewqard for every step forwards (left to right) the agent takes.

### Initialisation

If using Google colab you need to install packages - comment out lines below.

In [ ]:
#!apt install swig cmake ffmpeg
#!apt-get install -y xvfb x11-utils
#!pip install stable-baselines3[extra] pyglet box2d box2d-kengz
#!pip install pyvirtualdisplay PyOpenGL PyOpenGL-accelerate

For Google colab comment out this cell to make a virtual rendering canvas so render calls work (we still wont; see display!)

In [ ]:
#import pyvirtualdisplay
#
#_display = pyvirtualdisplay.Display(visible=False,  # use False with Xvfb
#                                    size=(1400, 900))
#_ = _display.start()

Import required packages.

In [ ]:
import gymnasium as gym
import stable_baselines3 as sb3
import torch

import bipedal_walker_ucd

import os
import pandas as pd # For data frames and data frame manipulation
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
import numpy as np # For general  numeric operations
from collections import deque

import gc # Import garbage collector

import imageio

import matplotlib.pyplot as plt
%matplotlib inline

### Define a Custom Wrapper and Evaluation Callback Function

We define a custom wrapper to allow experimantion twith different rewards in the Bipedal Walker environment. This adds potential extra reward terms:

- **Stability:** Encourage the hull to stay horizontal.
- **Target Speed:** Encaourage the agent to move at a target speed. The target speed can be set.
- **Smoothness:** Encourages similar action from one step to the next.

Each of these different com-oentnts of the reward can have a different weight associated with it. These are passed as parameters to the reward wrapper.
- `action_penalty_weight`
- `stability_weight`
- `speed_weight`
- `smoothness_weight`

When speed is given a non-zero weight a target speed should also be provided as `target_speed`.

In [ ]:
class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env, window=20,
                 action_penalty_weight = 0.0,
                 stability_weight=0.0,
                 smoothness_weight = 0.0,
                 speed_weight=0.0,
                 target_speed=1.5
                 ):
        super().__init__(env)
        self.window = window
        self.action_penalty_weight = action_penalty_weight
        self.stability_weight = stability_weight
        self.smoothness_weight = smoothness_weight
        self.speed_weight = speed_weight
        self.target_speed = target_speed

        self.prev_action = np.zeros(env.action_space.shape)
        self.hip1_history = deque(maxlen=window)
        self.hip2_history = deque(maxlen=window)

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)

        # --- Extract state ---
        hull = self.env.unwrapped.hull
        hull_angle   = hull.angle
        hull_vel_x   = hull.linearVelocity[0]
        hull_vel_y   = hull.linearVelocity[1]
        hull_height  = hull.position.y

        hip1_angle  = obs[4]
        hip2_angle  = obs[10]
        hip1_vel    = obs[5]
        hip2_vel    = obs[11]
        knee1_angle = obs[6]
        knee2_angle = obs[12]
        knee1_vel   = obs[7]
        knee2_vel   = obs[13]

        # --- Action Penalty ----
        # normalized to about -50.0 using heuristic, more optimal agent should spend less
        action_penalty = 0
        for a in action:
            action_penalty -= 0.00035 * bipedal_walker_ucd.MOTORS_TORQUE * np.clip(np.abs(a), 0, 1)
        action_penalty = action_penalty*self.action_penalty_weight

        # --- Stability ---
        stability = -abs(hull_angle) * self.stability_weight

        # --- Smoothness ---
        smoothness = -np.sum(np.abs(action - self.prev_action)) * self.smoothness_weight

        # --- Speed targeting ---
        speed_shaping = -abs(hull_vel_x - self.target_speed) * self.speed_weight

        # --- Total reward ---
        modified_reward = (
            base_reward
            + action_penalty
            + stability
            + smoothness
            + speed_shaping
        )

        # --- Reward components for TensorBoard ---
        info["reward_components"] = {
            "base":              base_reward,
            "action_penalty":    action_penalty,
            "stability":         stability,
            "smoothness":        smoothness,
            "speed_shaping":     speed_shaping,
            "total":             modified_reward,
        }

        self.prev_action = action.copy()
        return obs, modified_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.prev_action = np.zeros(self.env.action_space.shape)
        self.hip1_history.clear()
        self.hip2_history.clear()
        return self.env.reset(**kwargs)

The `VideoEvalCallback` writes out the components of the reward function defined above and also records videos of agent behaviour.

In [ ]:

class VideoEvalCallback(sb3.common.callbacks.BaseCallback):
    def __init__(
        self,
        eval_env: gym.Env,
        eval_freq: int = 10_000,
        n_eval_episodes: int = 1,
        video_folder: str = "./eval_videos",
        video_length: int = 1000,
        tensorboard_log: str = "./tb_logs",
        verbose: int = 1,
    ):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.video_folder = video_folder
        self.video_length = video_length
        self.writer = torch.utils.tensorboard.SummaryWriter(log_dir=tensorboard_log)
        self.eval_count = 0
        os.makedirs(video_folder, exist_ok=True)

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            self._run_eval()
        return True

    def _run_eval(self):
        self.eval_count += 1
        episode_rewards = []
        video_frames = None  # only keep episode 0 frames
        component_totals = {}

        for episode in range(self.n_eval_episodes):
            obs, _ = self.eval_env.reset()
            done = False
            total_reward = 0.0
            ep_frames = []
            ep_components = {}

            while not done:
                # Only render for episode 0 (video episode)
                if episode == 0:
                    frame = self.eval_env.render()
                    if frame is not None:
                        ep_frames.append(frame)

                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = self.eval_env.step(action)
                total_reward += reward
                done = terminated or truncated

                components = info.get("reward_components", {})
                for key, val in components.items():
                    ep_components[key] = ep_components.get(key, 0.0) + val

            episode_rewards.append(total_reward)

            for key, val in ep_components.items():
                component_totals[key] = component_totals.get(key, 0.0) + val

            if episode == 0:
                video_frames = ep_frames

            # Explicitly free ep_frames for non-video episodes
            del ep_frames

        mean_reward = np.mean(episode_rewards)
        std_reward  = np.std(episode_rewards)

        self.writer.add_scalar("eval/mean_reward", mean_reward, self.num_timesteps)
        self.writer.add_scalar("eval/std_reward",  std_reward,  self.num_timesteps)

        for key, total in component_totals.items():
            mean_val = total / self.n_eval_episodes
            self.writer.add_scalar(
                f"eval/reward_components/{key}", mean_val, self.num_timesteps
            )

        if self.verbose:
            print(f"\n[Eval @ {self.num_timesteps} steps]")
            print(f"  Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
            for key, total in component_totals.items():
                print(f"    {key:>20s}: {total / self.n_eval_episodes:.4f}")

        if video_frames:
            self._log_video_to_tensorboard(video_frames)

        # Free video frames after logging
        del video_frames

    def _log_video_to_tensorboard(self, frames: list):
        video_path = os.path.join(
            self.video_folder, f"eval_{self.eval_count}_step_{self.num_timesteps}.mp4"
        )
        imageio.mimsave(video_path, frames, fps=30)

        if self.verbose:
            print(f"  Video saved to {video_path}")

        video_array = np.array(frames, dtype=np.uint8)       # (T, H, W, C)
        video_array = video_array.transpose(0, 3, 1, 2)      # (T, C, H, W)
        video_array = np.expand_dims(video_array, axis=0)    # (1, T, C, H, W)
        video_tensor = video_array.astype(np.float32) / 255.0

        self.writer.add_video(
            "eval/agent_behaviour",
            video_tensor,
            global_step=self.num_timesteps,
            fps=30,
        )
        self.writer.flush()

        # Free the large arrays immediately
        del video_array, video_tensor

    def _on_training_end(self):
        self.writer.close()

### Task 1 Run A Basic Experiment
Create an instance of the **BipedalWalkerUCD** environment and add a `CustomRewardWrapper` defining the reward mix that will be used in the experiment - in this case set the weight of all non-base components to zero.

#### Create the Training Environment
Weight components are defined here for reuse throughout the notebook. It is also worth adding a `TimeLimt` wrapper to avoid very long runs during training. All weights are set to `0.0` for this experiment.

In [ ]:
action_penalty_weight = 0
symmetry_weight = 0
stability_weight = 0
smoothness_weight = 0
speed_weight = 0
target_speed = 1.5

We also define a set of utility values used throughout the experiment.

In [ ]:
time_limit = 5000
tb_log_path = './log_tb_bipedal_PPO/'
eval_log_path = './logs_bipedal_PPO/'
run_name = "Basic_Reward"
eval_video_path = './eval_videos/' + run_name + '/'
tb_log_name = run_name

Create the environment.

In [ ]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode=None)
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                         )
env = gym.wrappers.TimeLimit(env,
                                max_episode_steps = time_limit)

Explore the environment - view the action space and observation space.

In [ ]:
env.action_space

In [ ]:
env.observation_space

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [ ]:
agent = sb3.PPO('MlpPolicy',
                env,
                batch_size = 64,
                clip_range = 0.2,
                ent_coef = 0.001,
                gae_lambda =  0.95,
                gamma = 0.99,
                learning_rate = 0.00025,
                n_steps = 2048,
                n_epochs = 10,
                verbose = 2,
                tensorboard_log = tb_log_path
                )

Examine the actor and critic network architectures.

In [ ]:
print(agent.policy)

#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [ ]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode='rgb_array')
eval_env = CustomRewardWrapper(eval_env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                         )
eval_env = gym.wrappers.TimeLimit(eval_env,
                                max_episode_steps = time_limit)

eval_log_path = eval_log_path

video_callback = VideoEvalCallback(
    eval_env = eval_env,
    eval_freq = 100000,        # evaluate every 20k steps
    n_eval_episodes = 1,       # average reward over 3 episodes, video from first
    video_folder = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose=1,
)

#### Run the Agent Training Process
Train the model for a large number of timesteps (500,000 timesteps will probably work well).

In [ ]:
agent.learn(total_timesteps = 500000,
            callback = video_callback,
            tb_log_name = run_name)

Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal_PPO/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### Save the Agent
Save the trained agent.

In [ ]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [ ]:
del agent
env.close()
del env
eval_env.close()
del eval_env

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

### Task 2 Run A Custom Reward Experiment
Run an experiment using a version of the cutoms reward function with some of the extra compenets turned on to create a smoother gait.

#### Create the Training Environment
Weight components are defined here for reuse throughout the notebook. It is also worth adding a `TimeLimt` wrapper to avoid very long runs during training. Add non-zero weight values for some of thes components (values between 0.0 and 2.0 will work well).

In [ ]:
action_penalty_weight = # Add weight
stability_weight = # Add weight
speed_weight = # Add weight
smoothness_weight = # Add weight

We also define a set of utility values used throughout the experiment.

In [ ]:
time_limit = 5000
tb_log_path = './log_tb_bipedal_PPO/'
eval_log_path = './logs_bipedal_PPO/'
run_name = "Modified_Reward"
eval_video_path = './eval_videos/' + run_name + '/'
tb_log_name = run_name

Create the environment.

In [ ]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                         )
env = gym.wrappers.TimeLimit(env,
                                max_episode_steps = time_limit)

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [ ]:
agent = sb3.PPO('MlpPolicy',
                env,
                batch_size = 64,
                clip_range = 0.2,
                ent_coef = 0.001,
                gae_lambda =  0.95,
                gamma = 0.99,
                learning_rate = 0.00025,
                n_steps = 2048,
                n_epochs = 10,
                verbose = 2,
                tensorboard_log = tb_log_path
                )

#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [ ]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
eval_env = CustomRewardWrapper(eval_env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                         )
eval_env = gym.wrappers.TimeLimit(eval_env,
                                max_episode_steps = time_limit)

eval_log_path = eval_log_path

video_callback = VideoEvalCallback(
    eval_env = eval_env,
    eval_freq = 20000,        # evaluate every 20k steps
    n_eval_episodes = 3,       # average reward over 3 episodes, video from first
    video_folder = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose=1,
)

#### Run the Agent Training Process
Train the model for a large number of timesteps (500,000 timesteps will probably work well).

In [ ]:
agent.learn(total_timesteps = 500000,
            callback = video_callback,
            tb_log_name = run_name)

Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal_PPO/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### Save the Agent
Save the trained agent.

In [ ]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [ ]:
try:
    del agent
except NameError:
    pass
env.close()
del env
eval_env.close()
del eval_env
gc.collect()

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

### Task 3 A Custom Reward

Define new version of the `CustomRewardWrapper` that introduces a new compoent into the reward function. the parts of the state extracted are likely to be useful for this.

#### Create a New Custom Reward Wrapper

Add a new term to this new custom reward function.

In [ ]:
class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env, window=20,
                 action_penalty_weight = 1.0,
                 stability_weight=0.3,
                 smoothness_weight = 0.2,
                 speed_weight=0.5,
                 target_speed=1.5,
                 custom_component_weight = 0.0 # Add custom component weight here
                 ):
        super().__init__(env)
        self.window = window
        self.action_penalty_weight = action_penalty_weight
        self.stability_weight = stability_weight
        self.smoothness_weight = smoothness_weight
        self.speed_weight = speed_weight
        self.target_speed = target_speed
        self.custom_component_weight = custom_component_weight # Add custom component weight here

        self.prev_action = np.zeros(env.action_space.shape)
        self.hip1_history = deque(maxlen=window)
        self.hip2_history = deque(maxlen=window)

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)

        # --- Extract state ---
        hull = self.env.unwrapped.hull
        hull_angle   = hull.angle
        hull_vel_x   = hull.linearVelocity[0]
        hull_vel_y   = hull.linearVelocity[1]
        hull_height  = hull.position.y

        hip1_angle  = obs[4]
        hip2_angle  = obs[10]
        hip1_vel    = obs[5]
        hip2_vel    = obs[11]
        knee1_angle = obs[6]
        knee2_angle = obs[12]
        knee1_vel   = obs[7]
        knee2_vel   = obs[13]

        # --- Action Penalty ----
        # normalized to about -50.0 using heuristic, more optimal agent should spend less
        action_penalty = 0
        for a in action:
            action_penalty -= 0.00035 * bipedal_walker_ucd.MOTORS_TORQUE * np.clip(np.abs(a), 0, 1)
        action_penalty = action_penalty*self.action_penalty_weight

        # --- Stability ---
        stability = -abs(hull_angle) * self.stability_weight

        # --- Smoothness ---
        smoothness = -np.sum(np.abs(action - self.prev_action)) * self.smoothness_weight

        # --- Speed targeting ---
        speed_shaping = -abs(hull_vel_x - self.target_speed) * self.speed_weight

        # --- Custom Component ---
        custom_component_reward = 0.0 # Add custom component logic here (e.g., -abs(hull_height - target_height) * self.custom_component_weight)


        # --- Total reward ---
        modified_reward = (
            base_reward
            + action_penalty
            + stability
            + smoothness
            + speed_shaping
            + custom_component_reward # Add custom component here
        )

        # --- Reward components for TensorBoard ---
        info["reward_components"] = {
            "base":              base_reward,
            "action_penalty":    action_penalty,
            "stability":         stability,
            "smoothness":        smoothness,
            "speed_shaping":     speed_shaping,
            "custom_component":  custom_component_reward, # Add custom component here
            "total":             modified_reward,
        }

        self.prev_action = action.copy()
        return obs, modified_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.prev_action = np.zeros(self.env.action_space.shape)
        self.hip1_history.clear()
        self.hip2_history.clear()
        return self.env.reset(**kwargs)

#### Run A Basic Experiment
Create an instance of the **BipedalWalkerUCD** environment and add a `CustomRewardWrapper` defining the reward mix that will be used in the experiment.

#### Create the Training Environment
Weight components are defined here for reuse throughout the notebook. It is also worth adding a `TimeLimt` wrapper to avoid very long runs during training. All weights are set to `0.0` for this experiment.

In [ ]:
action_penalty_weight = 0.0 # Add weight
stability_weight = 0.0 # Add weight
speed_weight = 0.0 # Add weight
smoothness_weight = 0.0 # Add weight
custom_component_weight = 0.0 # Add custom component weight here

We also define a set of utility values used throughout the experiment.

In [ ]:
time_limit = 5000
tb_log_path = './log_tb_bipedal_PPO/'
eval_log_path = './logs_bipedal_PPO/'
run_name = "Custom_Reward"
eval_video_path = './eval_videos/' + run_name + '/'
tb_log_name = run_name

Create the environment.

In [ ]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                          smoothness_weight = smoothness_weight,
                          custom_component_weight = custom_component_weight # Add custom component here
                         )
env = gym.wrappers.TimeLimit(env,
                                max_episode_steps = time_limit)

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [ ]:
agent = sb3.PPO('MlpPolicy',
                env,
                batch_size = 64,
                clip_range = 0.2,
                ent_coef = 0.001,
                gae_lambda =  0.95,
                gamma = 0.99,
                learning_rate = 0.00025,
                n_steps = 2048,
                n_epochs = 10,
                verbose = 2,
                tensorboard_log = tb_log_path
                )

#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [ ]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode="rgb_array")
eval_env = CustomRewardWrapper(eval_env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                          custom_component_weight = custom_component_weight # Add custom component here
                         )
eval_env = gym.wrappers.TimeLimit(eval_env,
                                max_episode_steps = time_limit)

eval_log_path = eval_log_path

video_callback = VideoEvalCallback(
    eval_env = eval_env,
    eval_freq = 20000,        # evaluate every 20k steps
    n_eval_episodes = 3,       # average reward over 3 episodes, video from first
    video_folder = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose=1,
)

#### Run the Agent Training Process
Train the model for a large number of timesteps (500,000 timesteps will probably work well).

In [ ]:
agent.learn(total_timesteps = 500000,
            callback = video_callback,
            tb_log_name = run_name)

Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal_PPO/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### Save the Agent
Save the trained agent.

In [ ]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [ ]:
try:
    del agent
except NameError:
    pass
env.close()
del env
eval_env.close()
del eval_env
gc.collect()

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

### Task 4 Overall Reflection

*Write a short (no more than 500 words) reflection on the different agents you have trained, the performance of these agents, and how you might reflect on which one was best.*